# The transformer

Notebook 1 produced the text, notebook 2 turned it into token ids. This notebook builds the
model that will consume them — and, just as importantly, **checks that it is correct before
anyone spends hours training it.**

A wrong transformer does not crash. It trains, the loss goes down, and you find out
something was broken after the fact. So the last third of this notebook is assertions.

The classes themselves live in [`transformer/model.py`](../transformer/model.py) so that
notebook 4 can import exactly the model explained here — one definition, no chance of the
notebook and the training script drifting apart.


## Load the tokenizer

The tokenizer decides one number the model cannot get wrong: `vocab_size`. It sizes both
the embedding table at the input and the output layer, so it has to be *one past the largest
id the tokenizer can emit* — see notebook 2 for what happens when it is not.


In [1]:
import sys
from pathlib import Path

try:
    import minbpe  # already installed in this environment
except ModuleNotFoundError:
    here = Path.cwd()
    for repo_root in (here, *here.parents):
        if (repo_root / "minbpe" / "minbpe" / "base.py").exists():
            sys.path.insert(0, str(repo_root / "minbpe"))
            print("using minbpe clone at:", repo_root / "minbpe")
            break
    else:
        raise ModuleNotFoundError(
            "minbpe not found. Install it with "
            "`pip install git+https://github.com/karpathy/minbpe.git`"
        )
else:
    print("using installed minbpe:", Path(minbpe.__file__).parent)

# Make the tutorial's own `transformer` package importable from the Notebooks/ folder.
sys.path.insert(0, str(Path.cwd().parent))

using minbpe clone at: /Users/harshpujari/Documents/Work/Aexonic/Projects/llms/minbpe


In [2]:
from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.load(model_file="../output/tokenizer/my_tokenizer.model")


def get_vocab_size(tokenizer: RegexTokenizer) -> int:
    """Number of embedding rows the model needs: one past the largest usable id.

    Not len(vocab) + len(special_tokens). load() rebuilds .vocab with the special tokens
    already folded in, so that expression counts them twice - 1029 on a freshly trained
    tokenizer but 1034 on a reloaded one. The model would then be free to sample ids
    1029-1033, which no vocabulary entry covers, and generation would die with a
    ValueError inside decode().
    """
    largest = max(tokenizer.vocab)
    if tokenizer.special_tokens:
        largest = max(largest, *tokenizer.special_tokens.values())
    return largest + 1


vocab_size = get_vocab_size(tokenizer)
print(f"vocab_size = {vocab_size}")

vocab_size = 1029


## Configuration

Every number below is a real trade-off, not a magic constant:

| Setting | Value | What it buys, what it costs |
| --- | --- | --- |
| `block_size` | 256 | Longest context the model can see. Attention cost grows with its **square**. |
| `n_embd` | 384 | Width of the residual stream — how much each position can carry. |
| `n_head` | 6 | Attention heads. Must divide `n_embd`, giving `384 / 6 = 64` per head. |
| `n_layer` | 6 | Depth. More layers, more composition, harder to train. |
| `dropout` | 0.2 | Regularisation. Matters here: the corpus is small, so overfitting is the default outcome. |

`n_embd` and `n_head` are coupled — `head_size = n_embd // n_head` — which is why the model
raises rather than silently truncating if they do not divide evenly.


In [3]:
import torch

torch.manual_seed(3647)

block_size = 256
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

# Apple Silicon exposes "mps"; without checking for it a Mac silently trains on CPU.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device     = {device}")
print(f"head_size  = n_embd // n_head = {n_embd} // {n_head} = {n_embd // n_head}")

device     = mps
head_size  = n_embd // n_head = 384 // 6 = 64


<details>
<summary><b>Q&A — block_size, attention, and why it grows with the square (click to expand)</b></summary>

**Q: What is `block_size`? Is it like an LLM's context window?**

Yes. It's the maximum number of tokens the model can look at when predicting the next
one. It sizes the `position_embedding_table`, which has exactly `block_size` rows — one
per possible position/slot. Feed the model a sequence longer than `block_size` and there
is no row to look up for that position; the model raises instead of guessing.

**Q: What does "attention" actually mean?**

For every position, build a **query** ("what am I looking for?"). Every position (including
itself) offers a **key** ("what do I contain?"). Compare the query against every key to get
relevance scores, turn those into weights that sum to 1 (softmax), then take a weighted
average of every position's **value** ("what I pass on if attended to"). The result is a new
vector for that position — a blend of whichever other positions were relevant.

**Q: Why does attention cost grow with the square of `block_size`?**

Because computing relevance scores means comparing *every* query against *every* key — a
`(T, T)` grid of dot products (`weights = q @ k.transpose(-2, -1)` in `Head.forward`).
Double the sequence length `T` and you don't double the work, you quadruple it (`T²`).
That's the well-known quadratic cost of standard self-attention, and the reason long-context
models are expensive.

**Q: Concrete worked example — how does the dot product actually happen?**

Toy example, 4-dim word vectors, sentence "it ... cat ... sat":

1. **Project to Q/K/V.** Each word's embedding vector is multiplied by learned weight
   matrices `W_q`, `W_k`, `W_v` (this is `self.query(x)`, `self.key(x)`, `self.value(x)`).
   These matrices are the *only* learned parameters in attention — the dot product and
   softmax steps are fixed math.

   ```
   q_it  = x_it  @ W_q  = [1.0, 0.5]
   k_cat = x_cat @ W_k  = [1.0, 0.4]
   k_sat = x_sat @ W_k  = [0.1, 0.0]
   ```

2. **Dot product = similarity.** Multiply matching positions, sum them up:

   ```
   q_it . k_cat = (1.0*1.0) + (0.5*0.4) = 1.2
   q_it . k_sat = (1.0*0.1) + (0.5*0.0) = 0.1
   ```

   A bigger dot product means the query and key point in a more similar direction, i.e.
   "more relevant."

3. **Scale** by `1/sqrt(head_size)` to keep the numbers from growing huge as dimensions
   increase (`* k.shape[-1] ** -0.5` in the code) — pure stabilization:

   ```
   1.2/8 = 0.15,   0.1/8 = 0.0125    (head_size=64 -> sqrt=8, in the real model)
   ```

4. **Softmax** turns scores into weights that sum to 1:

   ```
   exp(0.15)=1.16, exp(0.0125)=1.01, sum=2.17
   weight_cat = 1.16/2.17 = 0.53
   weight_sat = 1.01/2.17 = 0.47
   ```

5. **Weighted sum of Values** = the new output vector for "it":

   ```
   output = 0.53 * v_cat + 0.47 * v_sat
   ```

   If `v_cat=[2.0, 0.0]` and `v_sat=[0.0, 2.0]`, output = `[1.06, 0.94]` — a blend, mostly
   "cat"'s content. This is `weights @ self.value(x)`, the last line of `Head.forward`.

**Pipeline, one line:** vector -> (learned `W_q/W_k/W_v`) -> query/key/value -> (dot product)
-> similarity score -> (scale + softmax) -> weight in [0,1] summing to 1 -> weighted blend of
value vectors -> new vector for that position. Learning happens entirely in `W_q`, `W_k`,
`W_v` being adjusted so this mechanism routes attention to the right places (e.g. "it" -> "cat").

</details>


## How the model works

Based on [Andrej Karpathy's implementation](https://github.com/karpathy/ng-video-lecture/blob/master/gpt.py).

Follow one batch of 6 tokens all the way through. Every shape below is for **this** config
(`n_embd=384`, `n_head=6`, `n_layer=6`, `vocab_size` from the tokenizer above), so you can
check them against the assertions at the bottom of the notebook.

| Stage | Output shape | What happened |
| --- | --- | --- |
| input ids | `(1, 6)` | 6 token ids |
| token embedding | `(1, 6, 384)` | each id looks up a row of a `(vocab_size, 384)` table |
| position embedding | `(6, 384)` | each *slot* 0–5 looks up a row of a `(256, 384)` table |
| sum of the two | `(1, 6, 384)` | broadcast-added: "which token" plus "where it sits" |
| ×6 transformer blocks | `(1, 6, 384)` | shape never changes — blocks refine, they do not resize |
| final layer norm | `(1, 6, 384)` | |
| output projection | `(1, 6, vocab_size)` | one score per vocabulary entry, per position |

### Step 1 — token and position embeddings

A token id is just an integer; it carries no meaning. The embedding table gives each id a
learned 384-dimensional vector. Position embeddings do the same for *slots* — attention has
no inherent notion of order, so without them "you owe me" and "me owe you" would be
identical inputs.

The position table has exactly `block_size = 256` rows. That is a hard ceiling: feed the
model a 257th token and there is no row to look up. The model raises instead of guessing.

### Step 2 — masked multi-head self-attention

Each position builds a **query** ("what am I looking for?"), a **key** ("what do I offer?")
and a **value** ("what do I pass on?"). Comparing every query against every key gives a
`(6, 6)` grid of relevance scores, which becomes a weighted average over the values.

Two details do the real work:

- **Masked.** Position 3 may attend to 0–3 but never 4 or 5. Enforced by setting future
  scores to `-inf` before the softmax, so they receive exactly zero weight. Without this,
  predicting the next token would be trivial — the answer is in the input — and the loss
  would look wonderful while the model learned nothing.
- **Multi-head.** Rather than one 384-wide attention, run 6 independent 64-wide ones and
  concatenate. Each head is free to specialise, and `6 × 64 = 384` so the width is unchanged.

### Step 3 — prediction head

The final linear layer maps each position's 384-dim vector to `vocab_size` logits. Softmax
turns those into a distribution over "what token comes next", and we sample from it.

Note this happens at **every** position, not just the last. A 6-token input produces 6
predictions, so one sequence gives 6 training signals instead of 1.


## Attention, written out

Below is attention implemented the explicit way: one module per head, the softmax and the
causal mask written by hand. This is the version to read and understand — every line maps to
a sentence in Step 2 above.

It is **not** the version we train with. The next section explains why, and then proves the
two are the same function.


In [4]:
import torch.nn as nn
from torch.nn import functional as F


class Head(nn.Module):
    """A single attention head, written out step by step."""

    def __init__(self, n_embd: int, head_size: int, block_size: int, dropout: float) -> None:
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # A lower-triangular matrix of 1s. Registered as a buffer so it moves with
        # .to(device) but is not a learned parameter.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, T, _ = x.shape                                    # (B, T, n_embd)
        k = self.key(x)                                      # (B, T, head_size)
        q = self.query(x)                                    # (B, T, head_size)

        # Every query against every key. Scaling by 1/sqrt(head_size) keeps the variance
        # at 1; without it the dot products grow with head_size, softmax saturates, and
        # gradients vanish before training starts.
        weights = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5      # (B, T, T)

        # The causal mask: -inf becomes exactly 0 after softmax.
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)

        return weights @ self.value(x)                        # (B, T, head_size)


class MultiHeadAttention(nn.Module):
    """Several heads in parallel, concatenated and projected back to n_embd."""

    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float) -> None:
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)]
        )
        self.projection = nn.Linear(head_size * n_head, n_embd)
        self.residual_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, n_embd)
        return self.residual_dropout(self.projection(out))

## The same thing, but fast

The code above is clear and slow. Two reasons:

1. **A Python loop over heads.** Six separate small matrix multiplications per layer, each
   with its own kernel launch, instead of one big batched one.
2. **A materialised `(T, T)` attention matrix**, written to memory and read back. It also
   means each head stores its own copy of an identical causal mask — 6 layers x 6 heads =
   **36 copies, 9.4 MB** of the same constant.

`transformer/model.py` fixes both: one `nn.Linear` produces query, key and value for all
heads at once, and `F.scaled_dot_product_attention` handles the scaling, masking and softmax
in a fused kernel. `is_causal=True` replaces the mask buffer entirely.

**Be honest about the size of the win.** Measured below at batch 16, context 256, the fused
version comes out at roughly **1.0-1.2x** on Apple MPS — which is to say, within measurement
noise. On CPU the same comparison gives 1.5-1.6x. On CUDA the gap is usually larger, because a
true flash-attention kernel gets selected; that is not measured here, so do not assume it.

So on this machine the fused path is **not** meaningfully faster. It is chosen for reasons that
hold regardless: 9.4 MB of duplicated mask buffers disappear, attention is no longer capped at
`block_size` by a buffer allocated at construction time, and it is the implementation you would
actually write for a real model. If the number printed below says 1.0x, that is expected.

Same parameters, same arithmetic — printed straight from the module so this cell can never
drift from the code that actually runs:


In [5]:
import inspect

from transformer.model import CausalSelfAttention, GPTLanguageModel

print(inspect.getsource(CausalSelfAttention.forward))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # (B, T, 3C) -> three (B, T, C) tensors -> each (B, n_head, T, head_size)
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        # Dropout only means anything when gradients are going to flow through it. Under
        # `no_grad` it would just add noise to a measurement, so it is switched off there
        # as well as in eval mode. That is also what keeps this portable: the MPS fused
        # kernel raises NotImplementedError for any non-zero dropout_p, and `no_grad` is
        # exactly when PyTorch picks that kernel.
        dropout_p = self.dropout if (self.training and torch.is_grad_enabled()) else 0.0

        # Scaling by 1/sqrt(head_size) and the causal mask are both handled intern

### Check: the fast version is the same function

"Should be equivalent" is not good enough for something everything else is built on. So:
copy the explicit version's weights into the fused one — the fused `qkv` is exactly the
per-head `query`/`key`/`value` matrices stacked — and compare outputs on the same input.

Both are put in `eval()` mode first. With dropout active they would disagree by design, and
the comparison would be meaningless.


In [6]:
import time

# Equivalence is checked on CPU: exact, reproducible, and independent of backend quirks.
B, T = 16, block_size
sample = torch.randn(B, T, n_embd)

loop_attention = MultiHeadAttention(n_embd, n_head, block_size, dropout).eval()
fused_attention = CausalSelfAttention(n_embd, n_head, dropout).eval()

# The fused qkv projection is the per-head q, k and v matrices stacked in that order.
with torch.no_grad():
    heads = loop_attention.heads
    fused_attention.qkv.weight.copy_(torch.cat([
        torch.cat([h.query.weight for h in heads]),
        torch.cat([h.key.weight for h in heads]),
        torch.cat([h.value.weight for h in heads]),
    ]))
    fused_attention.projection.weight.copy_(loop_attention.projection.weight)
    fused_attention.projection.bias.copy_(loop_attention.projection.bias)

with torch.no_grad():
    difference = (loop_attention(sample) - fused_attention(sample)).abs().max().item()

assert difference < 1e-5, f"fused attention disagrees with the explicit version: {difference}"
assert sum(p.numel() for p in loop_attention.parameters()) == \
       sum(p.numel() for p in fused_attention.parameters()), "parameter counts differ"


def synchronize(device):
    """Wait for queued work to finish.

    CUDA and MPS dispatch asynchronously: the Python call returns as soon as the work is
    *queued*, not when it is done. Timing without this measures how fast you can submit
    kernels, which makes any implementation that submits fewer of them look far better
    than it is. Getting this wrong is how you end up reporting a 23x speedup that is
    really 1.1x.
    """
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()


def time_forward_backward(module, repeats=10):
    x = torch.randn(B, T, n_embd, device=device, requires_grad=True)
    for _ in range(3):                       # warm-up, so we do not time lazy init
        module(x).sum().backward()
    synchronize(device)
    start = time.perf_counter()
    for _ in range(repeats):
        module(x).sum().backward()
    synchronize(device)
    return (time.perf_counter() - start) / repeats * 1e3


explicit_ms = time_forward_backward(loop_attention.to(device))
fused_ms = time_forward_backward(fused_attention.to(device))

mask_bytes = n_layer * n_head * block_size * block_size * 4
print(f"max absolute difference : {difference:.2e}  (float32 noise)")
print(f"parameters              : {sum(p.numel() for p in fused_attention.parameters()):,} in both")
print(f"forward+backward on {device:<4}: {explicit_ms:.1f} ms explicit -> {fused_ms:.1f} ms fused "
      f"({explicit_ms / fused_ms:.1f}x)")
print(f"causal mask buffers     : {mask_bytes / 1e6:.1f} MB explicit -> 0 MB fused")

max absolute difference : 2.38e-07  (float32 noise)
parameters              : 590,208 in both
forward+backward on mps : 30.8 ms explicit -> 28.3 ms fused (1.1x)
causal mask buffers     : 9.4 MB explicit -> 0 MB fused


## Building the model

Every hyperparameter is passed explicitly. The original version read `n_embd`, `block_size`,
`dropout` and `vocab_size` from module-level globals, which meant `GPTLanguageModel()` bound
to whatever those happened to be at call time — edit a config cell, re-run only the model
cell, and you get a model that half-agrees with the notebook. Passing them in makes that
impossible.


In [7]:
model = GPTLanguageModel(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout,
).to(device)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

11.53 M parameters


## Checking the model before training it

This is the cell that earns its keep. A transformer with a subtly broken component still
trains — the loss still falls — so these failures are invisible unless you look for them
deliberately. Four checks, all of them cheap:

1. **Shapes.** Logits must be `(batch, time, vocab_size)`. If the last dimension is not
   `vocab_size`, the model and tokenizer disagree about how many tokens exist.

2. **Initial loss ≈ `ln(vocab_size)`.** Before training, the model should be maximally
   uncertain — a uniform distribution over `vocab_size` options has cross-entropy
   `ln(vocab_size)` ≈ 6.94. Much *higher* means the initialisation is too large and the
   model starts out confidently wrong; much *lower* is worse, because it means information
   is leaking. This one number catches an enormous range of mistakes.

3. **Causal masking actually masks.** Change only the *last* token of a sequence. Every
   earlier position's logits must be bit-for-bit identical, because they are not allowed to
   see it. If they move, the model can read the future, next-token prediction becomes
   copying, and you get a beautiful loss curve from a model that has learned nothing.

4. **Generation stays in range.** Every sampled id must be decodable. This is the same
   `vocab_size` bug from notebook 2, caught here rather than mid-generation.


In [8]:
import math

model.eval()  # disable dropout so the checks are deterministic

# --- 1. shapes -------------------------------------------------------------
batch, sequence_length = 4, 6
inputs = torch.randint(0, vocab_size, (batch, sequence_length), device=device)
targets = torch.randint(0, vocab_size, (batch, sequence_length), device=device)

logits, loss = model(inputs, targets)
assert logits.shape == (batch, sequence_length, vocab_size), logits.shape
assert model(inputs)[1] is None, "loss should be None when no targets are given"

# --- 2. initial loss ~ ln(vocab_size) --------------------------------------
expected = math.log(vocab_size)
assert abs(loss.item() - expected) < 0.5, (
    f"initial loss {loss.item():.3f} is far from ln(vocab_size)={expected:.3f}; "
    "the initialisation is probably wrong"
)

# --- 3. causal masking ------------------------------------------------------
original = torch.randint(0, vocab_size, (1, 16), device=device)
edited = original.clone()
edited[0, -1] = (edited[0, -1] + 1) % vocab_size        # change ONLY the final token

with torch.no_grad():
    logits_original, _ = model(original)
    logits_edited, _ = model(edited)

leak = (logits_original[0, :-1] - logits_edited[0, :-1]).abs().max().item()
assert leak == 0.0, f"future leaked into earlier positions by {leak}"

# --- 4. generation stays inside the vocabulary ------------------------------
prompt = torch.tensor([tokenizer.encode("hey")], device=device)
generated = model.generate(prompt, max_new_tokens=32)
assert generated.shape == (1, prompt.shape[1] + 32)
assert int(generated.max()) < vocab_size, "sampled an id outside the vocabulary"
tokenizer.decode(generated[0].tolist())                 # must not raise

print(f"1. logits shape      {tuple(logits.shape)}")
print(f"2. initial loss      {loss.item():.3f}   (ln({vocab_size}) = {expected:.3f})")
print(f"3. causal mask       earlier positions moved by {leak}")
print(f"4. generation        {generated.shape[1]} ids, max {int(generated.max())} < {vocab_size}")
print("\nall checks passed")

1. logits shape      (4, 6, 1029)
2. initial loss      6.886   (ln(1029) = 6.936)
3. causal mask       earlier positions moved by 0.0
4. generation        35 ids, max 1028 < 1029

all checks passed


An untrained model produces noise, but it should be *decodable* noise. If
this looks like mojibake rather than random text, the tokenizer and the model disagree
somewhere.

In [9]:
print(repr(tokenizer.decode(generated[0].tolist())))

'hey joanceurn� gmute never\x17de�<|padding|>ful� acurn\x00 pr done� still honestly fix evening��ence] likeaking found ear whole'


## Where the parameters are

Worth a look before training: it tells you what you are actually spending capacity on. For a
small vocabulary and a narrow model, the embedding and output layers are a surprisingly large
share of the total.


In [10]:
def print_model_structure(module: torch.nn.Module, indent: str = "") -> None:
    """Print the module tree with a parameter count at each level."""
    for name, child in module.named_children():
        params = sum(p.numel() for p in child.parameters())
        print(f"{indent}|- {name}: {child.__class__.__name__} ({params:,} parameters)")
        print_model_structure(child, indent + "|  ")


print_model_structure(model)

|- token_embedding_table: Embedding (395,136 parameters)
|- position_embedding_table: Embedding (98,304 parameters)
|- blocks: Sequential (10,639,872 parameters)
|  |- 0: Block (1,773,312 parameters)
|  |  |- self_attention: CausalSelfAttention (590,208 parameters)
|  |  |  |- qkv: Linear (442,368 parameters)
|  |  |  |- projection: Linear (147,840 parameters)
|  |  |  |- residual_dropout: Dropout (0 parameters)
|  |  |- feed_forward: FeedForward (1,181,568 parameters)
|  |  |  |- net: Sequential (1,181,568 parameters)
|  |  |  |  |- 0: Linear (591,360 parameters)
|  |  |  |  |- 1: ReLU (0 parameters)
|  |  |  |  |- 2: Linear (590,208 parameters)
|  |  |  |  |- 3: Dropout (0 parameters)
|  |  |- layer_norm_1: LayerNorm (768 parameters)
|  |  |- layer_norm_2: LayerNorm (768 parameters)
|  |- 1: Block (1,773,312 parameters)
|  |  |- self_attention: CausalSelfAttention (590,208 parameters)
|  |  |  |- qkv: Linear (442,368 parameters)
|  |  |  |- projection: Linear (147,840 parameters)
|  

In [11]:
import pandas as pd

leaves = [
    {
        "Layer Name": name,
        "Type": module.__class__.__name__,
        "Parameters": sum(p.numel() for p in module.parameters()),
    }
    for name, module in model.named_modules()
    if not list(module.children())          # leaf modules only
]

stats = pd.DataFrame(leaves)
total = stats["Parameters"].sum()
stats["Share %"] = (100 * stats["Parameters"] / total).round(2)

print(f"{total:,} parameters across {len(stats)} leaf modules\n")
print(stats.groupby("Type", as_index=False)["Parameters"].sum()
           .assign(**{"Share %": lambda d: (100 * d["Parameters"] / total).round(2)})
           .sort_values("Parameters", ascending=False)
           .to_string(index=False))

11,530,245 parameters across 58 leaf modules

     Type  Parameters  Share %
   Linear    11026821    95.63
Embedding      493440     4.28
LayerNorm        9984     0.09
  Dropout           0     0.00
     ReLU           0     0.00


The model is defined and verified. `4_1_ModelTrainingAllBatches.ipynb`
imports it straight from `transformer/model.py` — the same code checked above — and trains it
on the token ids from notebook 2.